# Screen contacts for responsivity to images
Specifically designed for EmotionTask data <br>
**Prerequisites:** <br>
* *montage.csv
* *iEEG.csv
* *events.csv
<br>

Contacts drawn from a montage file <br>
*All* pairs of valid contacts from the montage file are identified <br>
Bipolar rereferenced signals are processed and screened for noise <br>
ERPs are generated to compare across regions

### Imports

In [1]:
# imports
import numpy as np
import pandas as pd
import os
import re
from itertools import combinations
from datetime import datetime, date, time, timedelta
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from scipy.ndimage import uniform_filter as unifilt
from scipy.signal import coherence, find_peaks

from iEEG_utils.loading import read_data
from iEEG_utils.processing import filtering
from iEEG_utils import utils

import ipywidgets as widgets

# instantiate a random number generator
rng = np.random.default_rng()

In [2]:
# use file selection dialog to choose folder with data
fstr = read_data.select_directory()
print(fstr)

C:\Users\jmile3\OneDrive - SCH\emotional_faces_data\915b69\EmotionTask


### Load data/metadata

In [3]:
# load each data file

montage = read_data.load_info(fstr,ftype="montage")
montage.name = montage.name.replace(" ","",regex=True)
origfs,data = read_data.load_iEEG(fstr,load_meta=True)
events = read_data.load_info(fstr,ftype="events")
data.columns = data.columns.str.replace(r"\s+", "", regex=True)

try:
    origtime = pd.to_datetime(data.time,format='%H:%M:%S.%f')
except AttributeError:
    origtime = pd.to_datetime(data.index,format='%H:%M:%S.%f')
    
origts = np.array(origtime-origtime[0])/ np.timedelta64(1, 's')

if origfs == None:
    origfs = 1/np.diff(origts).astype("float64").mean()
    print("original sampling frequency not declared, calculated as: "+str(origfs)+" Hz")


display(data.head())

,LT11,LT12,LE1,LE2,LE12,LE13,LC1,LC2,LC3,LC4,...,RG3,RG4,RG5,RB1,RB2,RB3,RB4,RB5,RB6,DC1
time,,,,,,,,,,,,,,,,,,,,,
11:11:25.356259,25.249142,21.359885,87.812533,61.727689,76.345930,76.278874,40.806169,73.931909,70.780270,38.459204,...,31.753589,25.517366,24.042131,26.724377,89.220712,43.823696,64.812272,43.622528,-3.786173,-7.675430
11:11:25.357259,25.919703,19.884649,81.844535,58.710162,76.949436,76.211818,46.572998,76.681211,73.529572,38.392148,...,32.424150,28.400781,26.925546,20.823436,83.923276,42.281405,64.678160,42.147292,-4.523790,-39.057709
11:11:25.358259,25.115029,16.934179,76.547099,55.692635,74.669527,75.407144,53.748007,76.547099,76.345930,39.800327,...,32.357094,30.479522,29.808960,14.989550,78.692896,38.526260,64.544047,41.275562,-3.786173,-71.110550
11:11:25.359259,24.310356,16.934179,68.433304,50.529311,70.176764,70.914382,55.089130,72.791954,72.523730,38.258035,...,29.339567,27.529051,28.333725,4.729959,69.841484,33.362936,61.526521,37.587474,-5.999026,-101.688156
11:11:25.360259,24.243299,16.129505,60.319510,46.774167,67.896855,68.634473,54.284456,68.232136,69.439147,36.715744,...,25.651479,24.578580,25.383254,-4.054397,58.777218,26.053816,57.033758,32.424150,-11.832911,-101.554044


In [4]:
# Align traces to image onsets

img_ts = origts[events.img_on_ix]

ch_list = montage.name.to_list()

pairs_list = utils.find_valid_pairs(ch_list)

# parameters for signal processing
resrate = 1024
lpfreq = 150

ts,bipolar,filtwins1 = filtering.bipolar_reref(data.loc[:,pairs_list[0]], origfs,
                                               resrate, lpfreq=lpfreq)

pre = -1
post = 2

t_ixs = np.array([np.where((ts>=(im_t+pre))&(ts<=(im_t+post))) for im_t in img_ts]).squeeze()
